# Notebook 11 - Clinical Validation Framework

We evaluate explanations for the readmission XGBoost model on three axes: fidelity
(does the explanation match the model), plausibility (does it match clinical literature),
and cross-method disagreement (do SHAP/LIME agree). The point is to adjudicate: when
methods disagree, fidelity + plausibility decide which to trust.

This runs all three horizons (30/60/90 days) in one loop and compares them. You do not
edit anything between horizons - just run the whole notebook top to bottom.

Core methods: TreeSHAP, KernelSHAP, LIME (all valid for trees). For 60/90 we retrain
XGBoost on the new label with the same hyperparameters. IG goes on the BiLSTM in a
separate notebook (trees are not differentiable).


## 1. Setup and config


In [1]:
# Run once if needed (Colab):
# !pip install shap lime scipy scikit-learn xgboost -q
import os, pickle, time, json, warnings
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
import xgboost as xgb
warnings.filterwarnings('ignore')

In [2]:
HORIZONS     = [30, 60, 90]   # run all three
N_EVAL       = 2000           # set ~300 for a fast first pass, then 2000
RANDOM_STATE = 42

def find_path(*c):
    for x in c:
        if os.path.exists(x): return x
    raise FileNotFoundError(c)
MROOT = find_path('../models/final_xgb pkl models/', '/content/')
CSV   = find_path('../Downloaded Files/bq-results-20260323-203006-1774297820102.csv',
                  '/content/bq-results-20260323-203006-1774297820102.csv')
OUT   = '../results/NB11/'; os.makedirs(OUT, exist_ok=True)
REF   = find_path('../results/NB11/clinical_reference_set.json', 'clinical_reference_set.json')
print('models:', MROOT)

models: /content/


## 2. Load model and data (once)

Horizon-independent: load the base model + artifacts, rebuild the data, encode with the
saved label mappings (not a fresh LabelEncoder), and build the chronological frame.


In [3]:
feature_cols = pickle.load(open(os.path.join(MROOT,'final_feature_cols.pkl'),'rb'))
split        = pickle.load(open(os.path.join(MROOT,'final_split_indices.pkl'),'rb'))
le_mappings  = pickle.load(open(os.path.join(MROOT,'final_le_mappings.pkl'),'rb'))
base_model   = pickle.load(open(os.path.join(MROOT,'xgboost_final_45f.pkl'),'rb'))  # 30d submitted model
train_end, val_end = split['train_end'], split['val_end']
cat_cols = ['gender','race','marital_status','language','insurance',
            'admission_location','discharge_location','admission_type']
cat_idx  = [feature_cols.index(c) for c in cat_cols]

In [4]:
df = pd.read_csv(CSV)
for c in ['marital_status','language','insurance','admission_location','discharge_location','race']:
    df[c] = df[c].fillna('UNKNOWN')
for c in ['num_lab_tests_24h','num_abnormal_labs','hemoglobin_min','wbc_max','creatinine_max',
          'sodium_min','sodium_max','potassium_min','potassium_max','glucose_min','glucose_max']:
    df[c] = df[c].fillna(df[c].median())
for c in ['days_since_last_discharge','num_admissions_last_30d','num_admissions_last_90d',
          'num_admissions_last_year','total_prior_admissions','recent_admission_flag','frequent_flyer_flag']:
    df[c] = df[c].fillna(0)
df = df.fillna(df.median(numeric_only=True))
for c in cat_cols:
    mapping = dict(le_mappings[c]['label_to_int'])
    mapped  = df[c].astype(str).map(mapping)
    if mapped.isna().any():
        fb = mapping.get('UNKNOWN', pd.Series(mapped.dropna()).mode().iloc[0])
        mapped = mapped.fillna(fb)
    df[c] = mapped.astype(int)
df['admittime'] = pd.to_datetime(df['admittime'])
df_sorted = df.sort_values('admittime').reset_index(drop=True)
X_all = df_sorted[feature_cols]
print('rows:', len(df_sorted))

rows: 406031


## 3. Reference set + all metric functions

Defined once, used for every horizon. Attribution methods, Krishna disagreement measures,
fidelity (faithfulness correlation + sparseness), and plausibility vs the clinical set.


In [5]:
!pip install lime
import shap
from lime.lime_tabular import LimeTabularExplainer

ref = json.load(open(REF))
high = [f['feature'] for f in ref['features'] if f['tier']=='high']
high_idx = [feature_cols.index(h) for h in high]
exp_sign = {f['feature']: f['expected_sign'] for f in ref['features']}
print(f'clinical reference set ({len(high)} high-tier):', high)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=6c2163ecba698b458250e40bfabf3cec95e893c73f4b44e63ba80617a9655be7
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime
clinical reference set (12 high-tier): ['los_days', 'los_hours', 'num_admissions_last_30d', 'num_admissions_last_90d', 'num_admissions_last_year', 'total_prior_admissions', 'num_diagnoses', 'admission_type', 'cci_chf', 'cci_cancer', 'cci_copd', 'cci_ckd']


In [6]:
# --- attribution methods (model + training frame passed in, since they change per horizon) ---
def get_attributions(model, X_train, X_eval):
    BG = shap.sample(X_train, 100, random_state=RANDOM_STATE)
    A = {}
    A['TreeSHAP'] = np.array(shap.TreeExplainer(model, data=BG,
                     feature_perturbation='interventional').shap_values(X_eval, check_additivity=False))
    f = lambda d: model.predict_proba(d)[:,1]
    A['KernelSHAP'] = np.array(shap.KernelExplainer(f, BG).shap_values(X_eval.values, nsamples=500, silent=True))
    lx = LimeTabularExplainer(X_train.values, feature_names=feature_cols, class_names=['no','yes'],
            categorical_features=cat_idx, discretize_continuous=True, mode='classification', random_state=RANDOM_STATE)
    L = np.zeros((len(X_eval), 45))
    for r in range(len(X_eval)):
        e = lx.explain_instance(X_eval.values[r], model.predict_proba, num_features=45, labels=(1,))
        for fi,w in e.as_map()[1]: L[r,fi] = w
    A['LIME'] = L
    return A, BG

In [7]:
# --- Axis C: Krishna disagreement measures ---
def _topk(a,k): return list(np.argsort(-np.abs(a))[:k])
def feature_agreement(a,b,k): return len(set(_topk(a,k))&set(_topk(b,k)))/k
def rank_agreement(a,b,k):
    ra,rb=_topk(a,k),_topk(b,k); return sum(ra[i]==rb[i] for i in range(k))/k
def sign_agreement(a,b,k):
    common=set(_topk(a,k))&set(_topk(b,k)); return sum(np.sign(a[i])==np.sign(b[i]) for i in common)/k
def rank_correlation(a,b,k):
    common=list(set(_topk(a,k))|set(_topk(b,k)))
    if len(common)<2: return np.nan
    return spearmanr(np.abs(a)[common], np.abs(b)[common]).correlation
MEASURES={'feature':feature_agreement,'rank':rank_agreement,'sign':sign_agreement,'rank_corr':rank_correlation}
PAIRS=[('TreeSHAP','KernelSHAP'),('TreeSHAP','LIME'),('KernelSHAP','LIME')]

In [8]:
# --- Axis A: fidelity ---
def faithfulness_corr(attr, X_eval_v, p_eval, model, baseline, n_runs=50, subset=8):
    rng=np.random.default_rng(RANDOM_STATE); n=len(X_eval_v)
    sums=np.zeros((n,n_runs)); drops=np.zeros((n,n_runs))
    for r in range(n_runs):
        mask=rng.random((n,45)).argsort(1)<subset
        Xp=X_eval_v.copy(); Xp[mask]=np.broadcast_to(baseline,(n,45))[mask]
        drops[:,r]=p_eval-model.predict_proba(Xp)[:,1]; sums[:,r]=(attr*mask).sum(1)
    cs=[np.corrcoef(sums[i],drops[i])[0,1] for i in range(n)]
    return float(np.nanmean([0 if np.isnan(c) else c for c in cs]))
def sparseness(attr):
    out=[]
    for a in attr:
        v=np.sort(np.abs(a)); n=len(v); cum=np.cumsum(v)
        out.append(0.0 if cum[-1]==0 else float(np.sum((2*np.arange(1,n+1)-n-1)*v)/(n*cum[-1])))
    return float(np.mean(out))

In [9]:
# --- Axis B: plausibility vs clinical reference ---
def plausibility(attr, K):
    imp=np.abs(attr).mean(0); signed=attr.mean(0)
    hit=set(np.argsort(-imp)[:K]) & set(high_idx)
    fa=len(hit)/min(K,len(high_idx)); sa=[]
    for idx in hit:
        es=exp_sign[feature_cols[idx]]
        if es=='null': continue
        sa.append(np.sign(signed[idx])==(1 if es=='positive' else -1))
    return fa, (float(np.mean(sa)) if sa else np.nan)

## 4. Run one horizon

Splits, retrains for 60/90, computes attributions (cached per horizon), and all three
axes. Saves per-horizon CSVs and returns a one-row summary.


In [10]:
def run_horizon(H):
    print(f'\n===== HORIZON {H} =====', flush=True)
    y = df_sorted[f'readmitted_{H}d']
    X_train, y_train = X_all.iloc[:train_end], y.iloc[:train_end]
    X_test,  y_test  = X_all.iloc[val_end:],  y.iloc[val_end:]
    assert len(X_test)>0, 'empty split'
    if H==30:
        model = base_model
    else:
        p = base_model.get_params(); p['scale_pos_weight']=(y_train==0).sum()/(y_train==1).sum()
        model = xgb.XGBClassifier(**p); model.fit(X_train, y_train)
    tr=roc_auc_score(y_train,model.predict_proba(X_train)[:,1])
    au=roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    print(f'base rate={y_test.mean():.3f} | train AUROC={tr:.3f} test AUROC={au:.3f} gap={tr-au:.3f}', flush=True)
    if H==30: assert abs(au-0.7183)<0.002, 'AUROC drift'
    assert tr-au < 0.10, 'train/test gap too large'

    X_eval = X_test.sample(N_EVAL, random_state=RANDOM_STATE)
    p_eval = model.predict_proba(X_eval)[:,1]
    cache = OUT+f'attrs_h{H}_n{N_EVAL}_v2.pkl'
    if os.path.exists(cache):
        A, BG = pickle.load(open(cache,'rb')); print('loaded cached attributions', flush=True)
    else:
        print('computing attributions (KernelSHAP is the slow one) ...', flush=True)
        t=time.time(); A, BG = get_attributions(model, X_train, X_eval)
        pickle.dump((A,BG), open(cache,'wb')); print(f'done {time.time()-t:.0f}s', flush=True)
    baseline = BG.values.mean(0)

    # Axis C
    drows=[]
    for (m1,m2) in PAIRS:
        for k in (5,10,15):
            for nm,fn in MEASURES.items():
                v=[fn(A[m1][i],A[m2][i],k) for i in range(N_EVAL)]
                drows.append({'horizon':H,'pair':f'{m1}-{m2}','K':k,'measure':nm,'mean':np.nanmean(v)})
    pd.DataFrame(drows).to_csv(OUT+f'disagreement_h{H}.csv',index=False)
    pc=[d['mean'] for d in drows if d['pair']=='TreeSHAP-KernelSHAP' and d['K']==10 and d['measure']=='feature'][0]
    print(f'positive control (TreeSHAP-KernelSHAP feat@10) = {pc:.3f} -> '+('PASS' if pc>=0.7 else 'CHECK'), flush=True)

    # Axis A
    rng=np.random.default_rng(RANDOM_STATE); rnd=rng.normal(size=A['TreeSHAP'].shape)
    Xev=X_eval.values
    fc_r=faithfulness_corr(rnd,Xev,p_eval,model,baseline)
    frows=[]
    for m in ['TreeSHAP','KernelSHAP','LIME']:
        fc=faithfulness_corr(A[m],Xev,p_eval,model,baseline)
        frows.append({'horizon':H,'method':m,'faithfulness_corr':round(fc,3),'vs_random':round(fc-fc_r,3),'sparseness':round(sparseness(A[m]),3)})
    pd.DataFrame(frows).to_csv(OUT+f'fidelity_h{H}.csv',index=False)

    # Axis B
    prows=[]
    for m in ['TreeSHAP','KernelSHAP','LIME']:
        for K in (5,10,15):
            fa,sa=plausibility(A[m],K)
            prows.append({'horizon':H,'method':m,'K':K,'feature_agree':round(fa,3),'sign_agree':None if np.isnan(sa) else round(sa,3)})
    pd.DataFrame(prows).to_csv(OUT+f'plausibility_h{H}.csv',index=False)

    # scorecard
    sc=[]
    for m in ['TreeSHAP','KernelSHAP','LIME']:
        fc=[r['faithfulness_corr'] for r in frows if r['method']==m][0]
        ph=[r for r in prows if r['method']==m and r['K']==10][0]
        sg=ph['sign_agree'] or 0
        sc.append({'horizon':H,'method':m,'fidelity':fc,'plaus_feat@10':ph['feature_agree'],'plaus_sign@10':sg,'trust_score':round((fc+sg)/2,3)})
    pd.DataFrame(sc).to_csv(OUT+f'trust_scorecard_h{H}.csv',index=False)
    return {'horizon':H,'base_rate':round(float(y_test.mean()),3),'test_auroc':round(au,3),
            'positive_control':round(pc,3),'scorecard':pd.DataFrame(sc)}

## 5. Run all horizons


In [11]:
results = {H: run_horizon(H) for H in HORIZONS}
print('\nall horizons done.')


===== HORIZON 30 =====
base rate=0.180 | train AUROC=0.753 test AUROC=0.718 gap=0.035
computing attributions (KernelSHAP is the slow one) ...


100%|===================| 1998/2000 [01:00<00:00]       

done 1462s
positive control (TreeSHAP-KernelSHAP feat@10) = 0.790 -> PASS

===== HORIZON 60 =====
base rate=0.248 | train AUROC=0.760 test AUROC=0.734 gap=0.026
computing attributions (KernelSHAP is the slow one) ...


100%|===================| 1997/2000 [00:35<00:00]       

done 1432s
positive control (TreeSHAP-KernelSHAP feat@10) = 0.780 -> PASS

===== HORIZON 90 =====
base rate=0.289 | train AUROC=0.765 test AUROC=0.743 gap=0.022
computing attributions (KernelSHAP is the slow one) ...


 98%|===================| 1958/2000 [00:36<00:00]       

done 1435s
positive control (TreeSHAP-KernelSHAP feat@10) = 0.791 -> PASS

all horizons done.


## 6. Compare across horizons

The headline: does SHAP stay the most trustworthy method as the prediction window widens?


In [12]:
summary = pd.DataFrame([{k:v for k,v in results[H].items() if k!='scorecard'} for H in HORIZONS])
summary.to_csv(OUT+'horizon_summary.csv', index=False)
print('Per-horizon model + control:'); display(summary)

allsc = pd.concat([results[H]['scorecard'] for H in HORIZONS], ignore_index=True)
allsc.to_csv(OUT+'scorecard_all_horizons.csv', index=False)
print('\nTrust score by method across horizons:')
display(allsc.pivot_table(index='method', columns='horizon', values='trust_score'))

best = {H: results[H]['scorecard'].sort_values('trust_score',ascending=False).iloc[0]['method'] for H in HORIZONS}
shap_family = ('TreeSHAP','KernelSHAP')
shap_always_wins = all(best[H] in shap_family for H in HORIZONS)
print('\nTop method per horizon:', best)
if shap_always_wins:
    print('STABLE FINDING: a SHAP method is most trustworthy at every horizon.')
    print('TreeSHAP and KernelSHAP swap the top spot within the SHAP family (they are near-tied);')
    print('both stay far above LIME across all horizons. Verdict = trust SHAP, not LIME.')
else:
    print('Top method is not consistently SHAP - investigate.')


Per-horizon model + control:


,horizon,base_rate,test_auroc,positive_control
0,30,0.180,0.718,0.790
1,60,0.248,0.734,0.780
2,90,0.289,0.743,0.791



Trust score by method across horizons:


horizon,30,60,90
method,,,
KernelSHAP,0.871,0.875,0.750
LIME,0.241,0.353,0.358
TreeSHAP,0.772,0.802,0.778



Top method per horizon: {30: 'KernelSHAP', 60: 'KernelSHAP', 90: 'TreeSHAP'}
STABLE FINDING: a SHAP method is most trustworthy at every horizon.
TreeSHAP and KernelSHAP swap the top spot within the SHAP family (they are near-tied);
both stay far above LIME across all horizons. Verdict = trust SHAP, not LIME.
